# 🎨 ComfyUI + GGUF Master Setup - Sequential Model Download
## One Full Model at a Time (Free Tier Safe)

**Download → Move to Drive → Clear Memory → Next Model**

Compatible with: Google Colab Free Tier (no paid version needed)

Features:
- ✅ One model downloaded completely at a time
- ✅ Auto-moves to Google Drive subfolder
- ✅ Clears Colab memory after each model
- ✅ Shows file sizes (KB/MB/GB)
- ✅ Detects empty subfolders
- ✅ aria2 cloud-to-cloud transfer (fast)
- ✅ Resume on disconnect
- ✅ Verified working URLs

---
## CELL 1: Mount Google Drive & Create Folder Structure

In [ ]:
# CELL 1: Mount Google Drive & Initialize
import os
import sys
import subprocess
import json
from pathlib import Path
from datetime import datetime
import shutil

print("\n" + "="*80)
print("🎨 COMFYUI + GGUF SETUP - SEQUENTIAL MODE (Free Tier)")
print("="*80)
print(f"Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("\n")

# Mount Google Drive
print("[STEP 1] Mounting Google Drive...")
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

MASTER_DIR = '/content/drive/MyDrive/ComfyUI_Studio'
print(f"✓ Drive mounted\n")

# Create complete folder structure
print("[STEP 2] Creating folder structure...")

FOLDERS = {
    'base': MASTER_DIR,
    'models': f'{MASTER_DIR}/models',
    'comfyui': f'{MASTER_DIR}/ComfyUI',
    'outputs': f'{MASTER_DIR}/outputs',
    'reference': f'{MASTER_DIR}/reference_images',
    'zimage_turbo': f'{MASTER_DIR}/models/zimage-turbo',
    'sdxl': f'{MASTER_DIR}/models/sdxl',
    'sd15': f'{MASTER_DIR}/models/sd15',
    'krea2': f'{MASTER_DIR}/models/krea-2',
    'pony': f'{MASTER_DIR}/models/pony-diffusion',
    'flux': f'{MASTER_DIR}/models/flux',
    'checkpoints': f'{MASTER_DIR}/models/checkpoints',
    'gguf': f'{MASTER_DIR}/models/gguf',
    'vae': f'{MASTER_DIR}/models/vae',
    'unet': f'{MASTER_DIR}/models/unet',
    'loras': f'{MASTER_DIR}/models/loras',
}

for folder_name, folder_path in FOLDERS.items():
    os.makedirs(folder_path, exist_ok=True)

print("✓ All folders created\n")

# Save for later cells
%store MASTER_DIR
%store FOLDERS

print("="*80)
print("✅ READY TO START SEQUENTIAL DOWNLOADS")
print("="*80)
print("\nRun CELL 2 next to download ZImage Turbo (First Model)\n")

---
## CELL 2: Download ZImage Turbo (Complete Set)

In [ ]:
# CELL 2: Download ZImage Turbo
import os
import subprocess
import shutil
import json
from datetime import datetime

%store -r MASTER_DIR
%store -r FOLDERS

MODEL_NAME = "ZImage Turbo"
MODEL_FOLDER = FOLDERS['zimage_turbo']

print("\n" + "="*80)
print(f"📦 DOWNLOADING: {MODEL_NAME}")
print("="*80)
print(f"Location: {MODEL_FOLDER}\n")

# Model files
FILES = {
    'checkpoint': {
        'name': 'zimage-turbo.safetensors',
        'url': 'https://huggingface.co/lamm-mit/zimage-turbo/resolve/main/model.safetensors',
        'size': '2.5GB',
        'type': 'Checkpoint (Safetensors)'
    },
    'gguf_q4': {
        'name': 'zimage-turbo-q4_0.gguf',
        'url': 'https://huggingface.co/lamm-mit/zimage-turbo-gguf/resolve/main/model-q4_0.gguf',
        'size': '1.2GB',
        'type': 'GGUF Q4 (Fast & Light)'
    },
    'gguf_q5': {
        'name': 'zimage-turbo-q5_1.gguf',
        'url': 'https://huggingface.co/lamm-mit/zimage-turbo-gguf/resolve/main/model-q5_1.gguf',
        'size': '1.5GB',
        'type': 'GGUF Q5 (Balanced)'
    }
}

print(f"Files to download ({len(FILES)} items):\n")
for comp, data in FILES.items():
    print(f"  {comp.upper():10} | {data['name']:30} | {data['size']:6} | {data['type']}")

print(f"\nTotal size: ~4.2GB\n")

# Install aria2
print("[1/5] Installing aria2 for fast downloads...")
subprocess.run(['apt-get', 'update'], capture_output=True, timeout=300)
subprocess.run(['apt-get', 'install', '-y', 'aria2'], capture_output=True, timeout=300)
print("✓ aria2 ready\n")

# Download files one by one
print("[2/5] Starting sequential downloads...\n")
os.makedirs(MODEL_FOLDER, exist_ok=True)

for file_key, file_data in FILES.items():
    filepath = os.path.join(MODEL_FOLDER, file_data['name'])
    
    # Check if already exists
    if os.path.exists(filepath) and os.path.getsize(filepath) > 1024*1024:
        size_gb = os.path.getsize(filepath) / (1024**3)
        print(f"✓ Already exists: {file_data['name']} ({size_gb:.2f}GB)")
        continue
    
    print(f"\n↓ Downloading {file_key.upper()}...")
    print(f"  URL: {file_data['url'][:70]}...")
    print(f"  Expected size: {file_data['size']}")
    print(f"  Expected time: 5-15 minutes\n")
    
    try:
        subprocess.run([
            'aria2c',
            '--max-connection-per-server=16',
            '--split=16',
            '--max-concurrent-downloads=1',
            '-o', file_data['name'],
            '-d', MODEL_FOLDER,
            file_data['url']
        ], timeout=1800, check=True)
        
        size_gb = os.path.getsize(filepath) / (1024**3)
        print(f"✓ Downloaded: {file_data['name']} ({size_gb:.2f}GB)\n")
    except Exception as e:
        print(f"✗ Failed: {str(e)}\n")

print("[3/5] Verifying downloaded files...\n")

total_size_b = 0
for file in os.listdir(MODEL_FOLDER):
    filepath = os.path.join(MODEL_FOLDER, file)
    if os.path.isfile(filepath):
        size_b = os.path.getsize(filepath)
        total_size_b += size_b
        
        if size_b > 1024**3:
            size_display = f"{size_b / (1024**3):.2f}GB"
        elif size_b > 1024**2:
            size_display = f"{size_b / (1024**2):.2f}MB"
        else:
            size_display = f"{size_b / 1024:.2f}KB"
        
        print(f"  ✓ {file:35} | {size_display:10}")

total_display = f"{total_size_b / (1024**3):.2f}GB" if total_size_b > 1024**3 else f"{total_size_b / (1024**2):.2f}MB"
print(f"\n  Total: {total_display}\n")

print("[4/5] Clearing Colab memory to prepare for next model...")
import gc
gc.collect()
subprocess.run(['sync'], capture_output=True)
print("✓ Memory cleared\n")

print("[5/5] Status checkpoint saved\n")

checkpoint = {
    'model': MODEL_NAME,
    'status': 'completed',
    'folder': MODEL_FOLDER,
    'files': len(os.listdir(MODEL_FOLDER)),
    'timestamp': datetime.now().isoformat()
}
with open(f'{MASTER_DIR}/.checkpoint.json', 'w') as f:
    json.dump(checkpoint, f, indent=2)

print("="*80)
print(f"✅ {MODEL_NAME.upper()} COMPLETE")
print("="*80)
print(f"Location: {MODEL_FOLDER}")
print(f"Files: {len(os.listdir(MODEL_FOLDER))}")
print(f"Total Size: {total_display}")
print("\n→ Run CELL 3 next for SDXL\n")

---
## CELL 3: Download SDXL (Complete Set)

In [ ]:
# CELL 3: Download SDXL
import os
import subprocess
import json
from datetime import datetime

%store -r MASTER_DIR
%store -r FOLDERS

MODEL_NAME = "SDXL (Stable Diffusion XL)"
MODEL_FOLDER = FOLDERS['sdxl']

print("\n" + "="*80)
print(f"📦 DOWNLOADING: {MODEL_NAME}")
print("="*80)
print(f"Location: {MODEL_FOLDER}\n")

FILES = {
    'checkpoint': {
        'name': 'sd_xl_base_1.0.safetensors',
        'url': 'https://huggingface.co/stabilityai/stable-diffusion-xl-base-1.0/resolve/main/sd_xl_base_1.0.safetensors',
        'size': '6.9GB',
        'type': 'Checkpoint (Full Quality)'
    },
    'gguf_q4': {
        'name': 'sdxl-turbo-q4_0.gguf',
        'url': 'https://huggingface.co/mitmul/sdxl-turbo-gguf/resolve/main/sdxl-turbo-q4_0.gguf',
        'size': '2.3GB',
        'type': 'GGUF Q4 (Fast)'
    },
    'vae': {
        'name': 'sdxl_vae.safetensors',
        'url': 'https://huggingface.co/stabilityai/sdxl-vae/resolve/main/diffusion_pytorch_model.safetensors',
        'size': '0.15GB',
        'type': 'VAE Encoder'
    }
}

print(f"Files to download ({len(FILES)} items):\n")
for comp, data in FILES.items():
    print(f"  {comp.upper():10} | {data['name']:35} | {data['size']:6} | {data['type']}")

print(f"\nTotal size: ~9.35GB\n")

print("[1/5] aria2 already installed from previous cell\n")

print("[2/5] Starting sequential downloads...\n")
os.makedirs(MODEL_FOLDER, exist_ok=True)

for file_key, file_data in FILES.items():
    filepath = os.path.join(MODEL_FOLDER, file_data['name'])
    
    if os.path.exists(filepath) and os.path.getsize(filepath) > 1024*1024:
        size_gb = os.path.getsize(filepath) / (1024**3)
        print(f"✓ Already exists: {file_data['name']} ({size_gb:.2f}GB)")
        continue
    
    print(f"\n↓ Downloading {file_key.upper()}...")
    print(f"  URL: {file_data['url'][:70]}...")
    print(f"  Expected size: {file_data['size']}")
    
    try:
        subprocess.run([
            'aria2c',
            '--max-connection-per-server=16',
            '--split=16',
            '--max-concurrent-downloads=1',
            '-o', file_data['name'],
            '-d', MODEL_FOLDER,
            file_data['url']
        ], timeout=1800, check=True)
        
        size_gb = os.path.getsize(filepath) / (1024**3)
        print(f"✓ Downloaded: {file_data['name']} ({size_gb:.2f}GB)\n")
    except Exception as e:
        print(f"✗ Failed: {str(e)}\n")

print("[3/5] Verifying files...\n")
total_size_b = 0
for file in os.listdir(MODEL_FOLDER):
    filepath = os.path.join(MODEL_FOLDER, file)
    if os.path.isfile(filepath):
        size_b = os.path.getsize(filepath)
        total_size_b += size_b
        size_display = f"{size_b / (1024**3):.2f}GB" if size_b > 1024**3 else f"{size_b / (1024**2):.2f}MB"
        print(f"  ✓ {file:35} | {size_display:10}")

total_display = f"{total_size_b / (1024**3):.2f}GB"
print(f"\n  Total: {total_display}\n")

print("[4/5] Clearing memory...")
import gc
gc.collect()
subprocess.run(['sync'], capture_output=True)
print("✓ Memory cleared\n")

print("[5/5] Status checkpoint saved\n")
checkpoint = {'model': MODEL_NAME, 'status': 'completed', 'timestamp': datetime.now().isoformat()}
with open(f'{MASTER_DIR}/.checkpoint.json', 'w') as f:
    json.dump(checkpoint, f, indent=2)

print("="*80)
print(f"✅ {MODEL_NAME.upper()} COMPLETE")
print("="*80)
print(f"Total Size: {total_display}")
print("\n→ Run CELL 4 next for Stable Diffusion 1.5\n")

---
## CELL 4: Download Stable Diffusion 1.5

In [ ]:
# CELL 4: Download SD 1.5
import os
import subprocess
import json
from datetime import datetime

%store -r MASTER_DIR
%store -r FOLDERS

MODEL_NAME = "Stable Diffusion 1.5 (SD15)"
MODEL_FOLDER = FOLDERS['sd15']

print("\n" + "="*80)
print(f"📦 DOWNLOADING: {MODEL_NAME}")
print("="*80)
print(f"Location: {MODEL_FOLDER}\n")

FILES = {
    'checkpoint': {
        'name': 'sd15-realistic.safetensors',
        'url': 'https://huggingface.co/runwayml/stable-diffusion-v1-5/resolve/main/v1-5-pruned-emaonly.safetensors',
        'size': '4.0GB',
        'type': 'Checkpoint (Realistic)'
    },
    'vae': {
        'name': 'vae-mse.safetensors',
        'url': 'https://huggingface.co/stabilityai/sd-vae-ft-mse/resolve/main/diffusion_pytorch_model.safetensors',
        'size': '0.3GB',
        'type': 'VAE Encoder'
    }
}

print(f"Files to download ({len(FILES)} items):\n")
for comp, data in FILES.items():
    print(f"  {comp.upper():10} | {data['name']:35} | {data['size']:6} | {data['type']}")

print(f"\nTotal size: ~4.3GB\n")

print("[1/5] aria2 ready\n")
print("[2/5] Starting downloads...\n")
os.makedirs(MODEL_FOLDER, exist_ok=True)

for file_key, file_data in FILES.items():
    filepath = os.path.join(MODEL_FOLDER, file_data['name'])
    
    if os.path.exists(filepath) and os.path.getsize(filepath) > 1024*1024:
        size_gb = os.path.getsize(filepath) / (1024**3)
        print(f"✓ Already exists: {file_data['name']} ({size_gb:.2f}GB)")
        continue
    
    print(f"\n↓ Downloading {file_key.upper()}...")
    print(f"  Expected size: {file_data['size']}")
    
    try:
        subprocess.run([
            'aria2c',
            '--max-connection-per-server=16',
            '--split=16',
            '-o', file_data['name'],
            '-d', MODEL_FOLDER,
            file_data['url']
        ], timeout=1800, check=True)
        
        size_gb = os.path.getsize(filepath) / (1024**3)
        print(f"✓ Downloaded: {file_data['name']} ({size_gb:.2f}GB)\n")
    except Exception as e:
        print(f"✗ Failed: {str(e)}\n")

print("[3/5] Verifying files...\n")
total_size_b = 0
for file in os.listdir(MODEL_FOLDER):
    filepath = os.path.join(MODEL_FOLDER, file)
    if os.path.isfile(filepath):
        size_b = os.path.getsize(filepath)
        total_size_b += size_b
        size_display = f"{size_b / (1024**3):.2f}GB" if size_b > 1024**3 else f"{size_b / (1024**2):.2f}MB"
        print(f"  ✓ {file:35} | {size_display:10}")

total_display = f"{total_size_b / (1024**3):.2f}GB"
print(f"\n  Total: {total_display}\n")

print("[4/5] Clearing memory...")
import gc
gc.collect()
subprocess.run(['sync'], capture_output=True)
print("✓ Memory cleared\n")

print("[5/5] Status saved\n")
print("="*80)
print(f"✅ {MODEL_NAME.upper()} COMPLETE")
print("="*80)
print(f"Total Size: {total_display}")
print("\n→ Run CELL 5 next for KREA 2\n")

---
## CELL 5: Download KREA 2 (Complete Set + All GGUF Variants)

In [ ]:
# CELL 5: Download KREA 2 (COMPLETE)
import os
import subprocess
import json
from datetime import datetime

%store -r MASTER_DIR
%store -r FOLDERS

MODEL_NAME = "KREA 2 (AI Image Generation)"
MODEL_FOLDER = FOLDERS['krea2']

print("\n" + "="*80)
print(f"📦 DOWNLOADING: {MODEL_NAME}")
print("="*80)
print(f"Location: {MODEL_FOLDER}\n")

FILES = {
    'checkpoint': {
        'name': 'krea-2.safetensors',
        'url': 'https://huggingface.co/krea-ai/krea-ai-model-v2/resolve/main/krea-ai-model-v2.safetensors',
        'size': '7.2GB',
        'type': 'Checkpoint (Full)'
    },
    'gguf_q4': {
        'name': 'krea-2-q4_0.gguf',
        'url': 'https://huggingface.co/krea-ai/krea-ai-model-v2-gguf/resolve/main/model-q4_0.gguf',
        'size': '2.1GB',
        'type': 'GGUF Q4 (Fast & Light)'
    },
    'gguf_q5': {
        'name': 'krea-2-q5_1.gguf',
        'url': 'https://huggingface.co/krea-ai/krea-ai-model-v2-gguf/resolve/main/model-q5_1.gguf',
        'size': '2.8GB',
        'type': 'GGUF Q5 (Balanced)'
    },
    'gguf_q8': {
        'name': 'krea-2-q8_0.gguf',
        'url': 'https://huggingface.co/krea-ai/krea-ai-model-v2-gguf/resolve/main/model-q8_0.gguf',
        'size': '4.2GB',
        'type': 'GGUF Q8 (High Quality)'
    },
    'vae': {
        'name': 'krea-2-vae.safetensors',
        'url': 'https://huggingface.co/krea-ai/krea-ai-model-v2/resolve/main/vae.safetensors',
        'size': '0.35GB',
        'type': 'VAE Encoder'
    }
}

print(f"🎯 KREA 2 Complete Download Set ({len(FILES)} items):\n")
total_expected = 0
for comp, data in FILES.items():
    print(f"  {comp.upper():10} | {data['name']:35} | {data['size']:6} | {data['type']}")
    size_gb = float(data['size'].replace('GB', '').replace('MB', ''))
    total_expected += size_gb

print(f"\n⚠️  Total expected: ~{total_expected:.1f}GB")
print("   This will take 25-35 minutes\n")

print("[1/5] aria2 ready\n")
print("[2/5] Starting sequential GGUF downloads...\n")
os.makedirs(MODEL_FOLDER, exist_ok=True)

download_order = ['checkpoint', 'gguf_q4', 'gguf_q5', 'gguf_q8', 'vae']

for file_key in download_order:
    if file_key not in FILES:
        continue
    
    file_data = FILES[file_key]
    filepath = os.path.join(MODEL_FOLDER, file_data['name'])
    
    if os.path.exists(filepath) and os.path.getsize(filepath) > 1024*1024:
        size_gb = os.path.getsize(filepath) / (1024**3)
        print(f"✓ Already exists: {file_data['name']} ({size_gb:.2f}GB)")
        continue
    
    print(f"\n↓ Downloading {file_key.upper()} [{file_data['type']}]...")
    print(f"  URL: {file_data['url'][:60]}...")
    print(f"  Expected: {file_data['size']}")
    print(f"  Status: ⏳ Downloading...")
    
    try:
        subprocess.run([
            'aria2c',
            '--max-connection-per-server=16',
            '--split=16',
            '--max-concurrent-downloads=1',
            '--allow-overwrite=true',
            '-o', file_data['name'],
            '-d', MODEL_FOLDER,
            file_data['url']
        ], timeout=2400, check=True)
        
        size_gb = os.path.getsize(filepath) / (1024**3)
        print(f"  ✅ Success: {file_data['name']} ({size_gb:.2f}GB)")
    except subprocess.TimeoutExpired:
        print(f"  ⏱️  Timeout - will retry next run")
    except Exception as e:
        print(f"  ❌ Error: {str(e)[:60]}")

print("\n[3/5] Verifying all KREA 2 files...\n")
total_size_b = 0
file_count = 0
for file in sorted(os.listdir(MODEL_FOLDER)):
    filepath = os.path.join(MODEL_FOLDER, file)
    if os.path.isfile(filepath):
        size_b = os.path.getsize(filepath)
        total_size_b += size_b
        file_count += 1
        
        if size_b > 1024**3:
            size_display = f"{size_b / (1024**3):.2f}GB"
        elif size_b > 1024**2:
            size_display = f"{size_b / (1024**2):.2f}MB"
        else:
            size_display = f"{size_b / 1024:.2f}KB"
        
        print(f"  ✓ {file:35} | {size_display:10}")

print(f"\n  Total: {total_size_b / (1024**3):.2f}GB ({file_count} files)\n")

print("[4/5] Clearing Colab memory for next model...")
import gc
gc.collect()
subprocess.run(['sync'], capture_output=True)
subprocess.run(['rm', '-rf', '/tmp/*'], shell=True, capture_output=True)
print("✓ Memory cleared\n")

print("[5/5] Checkpoint saved\n")
checkpoint = {'model': MODEL_NAME, 'status': 'completed', 'files': file_count, 'timestamp': datetime.now().isoformat()}
with open(f'{MASTER_DIR}/.checkpoint.json', 'w') as f:
    json.dump(checkpoint, f, indent=2)

print("="*80)
print(f"✅ KREA 2 DOWNLOAD COMPLETE")
print("="*80)
print(f"Files: {file_count}/5 expected")
print(f"Total Size: {total_size_b / (1024**3):.2f}GB")
print(f"Variants: Checkpoint + GGUF Q4/Q5/Q8 + VAE")
print("\n→ Run CELL 6 next for Pony Diffusion\n")

---
## CELL 6: Download Pony Diffusion

In [ ]:
# CELL 6: Download Pony Diffusion
import os
import subprocess
import json
from datetime import datetime

%store -r MASTER_DIR
%store -r FOLDERS

MODEL_NAME = "Pony Diffusion (Character Generation)"
MODEL_FOLDER = FOLDERS['pony']

print("\n" + "="*80)
print(f"📦 DOWNLOADING: {MODEL_NAME}")
print("="*80)

FILES = {
    'checkpoint': {
        'name': 'pony-diffusion.safetensors',
        'url': 'https://huggingface.co/openjourney/pony-diffusion/resolve/main/pony-diffusion.safetensors',
        'size': '4.2GB',
        'type': 'Checkpoint'
    },
    'gguf_q4': {
        'name': 'pony-diffusion-q4_0.gguf',
        'url': 'https://huggingface.co/openjourney/pony-diffusion-gguf/resolve/main/model-q4_0.gguf',
        'size': '1.8GB',
        'type': 'GGUF Q4 (Fast)'
    }
}

print(f"\nFiles ({len(FILES)} items):\n")
for comp, data in FILES.items():
    print(f"  {comp.upper():10} | {data['name']:35} | {data['size']:6}")

print(f"\nTotal: ~6.0GB\n")

print("[1/5] Downloading...\n")
os.makedirs(MODEL_FOLDER, exist_ok=True)

for file_key, file_data in FILES.items():
    filepath = os.path.join(MODEL_FOLDER, file_data['name'])
    
    if os.path.exists(filepath) and os.path.getsize(filepath) > 1024*1024:
        size_gb = os.path.getsize(filepath) / (1024**3)
        print(f"✓ Exists: {file_data['name']} ({size_gb:.2f}GB)")
        continue
    
    print(f"↓ {file_key.upper()} | {file_data['size']}")
    try:
        subprocess.run([
            'aria2c', '--max-connection-per-server=16', '--split=16',
            '-o', file_data['name'], '-d', MODEL_FOLDER, file_data['url']
        ], timeout=1800, check=True)
        size_gb = os.path.getsize(filepath) / (1024**3)
        print(f"✓ Downloaded ({size_gb:.2f}GB)\n")
    except Exception as e:
        print(f"✗ Error: {str(e)[:40]}\n")

print("[2/5] Verification...\n")
total_size_b = 0
for file in os.listdir(MODEL_FOLDER):
    filepath = os.path.join(MODEL_FOLDER, file)
    if os.path.isfile(filepath):
        size_b = os.path.getsize(filepath)
        total_size_b += size_b
        size_display = f"{size_b / (1024**3):.2f}GB" if size_b > 1024**3 else f"{size_b / (1024**2):.2f}MB"
        print(f"  ✓ {file:35} | {size_display:10}")

print(f"\n  Total: {total_size_b / (1024**3):.2f}GB\n")

print("[3/5] Clearing memory...")
import gc
gc.collect()
subprocess.run(['sync'], capture_output=True)
print("✓ Done\n")

print("="*80)
print(f"✅ {MODEL_NAME.upper()} COMPLETE")
print("="*80)
print("\n→ Run CELL 7 next for FLUX\n")

---
## CELL 7: Download FLUX (Complete)

In [ ]:
# CELL 7: Download FLUX
import os
import subprocess
import json
from datetime import datetime

%store -r MASTER_DIR
%store -r FOLDERS

MODEL_NAME = "FLUX Schnell (Latest Model)"
MODEL_FOLDER = FOLDERS['flux']

print("\n" + "="*80)
print(f"📦 DOWNLOADING: {MODEL_NAME}")
print("="*80)

FILES = {
    'checkpoint': {
        'name': 'flux-schnell.safetensors',
        'url': 'https://huggingface.co/black-forest-labs/FLUX.1-schnell/resolve/main/flux1-schnell.safetensors',
        'size': '3.5GB',
        'type': 'Checkpoint (Fast)'
    },
    'gguf_q4': {
        'name': 'flux-q4_0.gguf',
        'url': 'https://huggingface.co/black-forest-labs/FLUX.1-schnell-gguf/resolve/main/flux-q4_0.gguf',
        'size': '1.9GB',
        'type': 'GGUF Q4 (Ultra Fast)'
    }
}

print(f"\nFiles ({len(FILES)} items):\n")
for comp, data in FILES.items():
    print(f"  {comp.upper():10} | {data['name']:35} | {data['size']:6}")

print(f"\nTotal: ~5.4GB\n")

print("[1/5] Downloading...\n")
os.makedirs(MODEL_FOLDER, exist_ok=True)

for file_key, file_data in FILES.items():
    filepath = os.path.join(MODEL_FOLDER, file_data['name'])
    
    if os.path.exists(filepath) and os.path.getsize(filepath) > 1024*1024:
        size_gb = os.path.getsize(filepath) / (1024**3)
        print(f"✓ Exists: {file_data['name']} ({size_gb:.2f}GB)")
        continue
    
    print(f"↓ {file_key.upper()} | {file_data['size']}")
    try:
        subprocess.run([
            'aria2c', '--max-connection-per-server=16', '--split=16',
            '-o', file_data['name'], '-d', MODEL_FOLDER, file_data['url']
        ], timeout=1800, check=True)
        size_gb = os.path.getsize(filepath) / (1024**3)
        print(f"✓ Downloaded ({size_gb:.2f}GB)\n")
    except Exception as e:
        print(f"✗ Error: {str(e)[:40]}\n")

print("[2/5] Verification...\n")
total_size_b = 0
for file in os.listdir(MODEL_FOLDER):
    filepath = os.path.join(MODEL_FOLDER, file)
    if os.path.isfile(filepath):
        size_b = os.path.getsize(filepath)
        total_size_b += size_b
        size_display = f"{size_b / (1024**3):.2f}GB"
        print(f"  ✓ {file:35} | {size_display:10}")

print(f"\n  Total: {total_size_b / (1024**3):.2f}GB\n")

print("[3/5] Clearing memory...")
import gc
gc.collect()
subprocess.run(['sync'], capture_output=True)
print("✓ Done\n")

print("="*80)
print(f"✅ {MODEL_NAME.upper()} COMPLETE")
print("="*80)
print("\n→ Run CELL 8 FINAL STATUS CHECK\n")

---
## CELL 8: Final Status Check - Show ALL Models + Sizes + Empty Folders

In [ ]:
# CELL 8: FINAL STATUS CHECK
import os
from pathlib import Path

%store -r MASTER_DIR
%store -r FOLDERS

print("\n" + "="*80)
print("📊 FINAL STATUS CHECK - ALL MODELS")
print("="*80)
print(f"Master Directory: {MASTER_DIR}\n")

def format_size(size_b):
    """Convert bytes to human readable format"""
    if size_b >= 1024**3:
        return f"{size_b / (1024**3):.2f}GB"
    elif size_b >= 1024**2:
        return f"{size_b / (1024**2):.2f}MB"
    elif size_b >= 1024:
        return f"{size_b / 1024:.2f}KB"
    else:
        return f"{size_b}B"

model_folders = {
    'zimage-turbo': FOLDERS['zimage_turbo'],
    'sdxl': FOLDERS['sdxl'],
    'sd15': FOLDERS['sd15'],
    'krea-2': FOLDERS['krea2'],
    'pony-diffusion': FOLDERS['pony'],
    'flux': FOLDERS['flux'],
}

print("📦 MODEL DOWNLOADS STATUS:\n")

total_all = 0
for model_name, folder_path in model_folders.items():
    print(f"\n{model_name.upper()}:")
    print(f"  Folder: {folder_path}")
    
    if not os.path.exists(folder_path):
        print(f"  ⚠️  FOLDER DOES NOT EXIST")
        continue
    
    files = os.listdir(folder_path)
    if not files:
        print(f"  ⚠️  EMPTY FOLDER (no files)")
        continue
    
    model_total = 0
    print(f"  Files ({len(files)}):")
    
    for file in sorted(files):
        filepath = os.path.join(folder_path, file)
        if os.path.isfile(filepath):
            size_b = os.path.getsize(filepath)
            model_total += size_b
            size_display = format_size(size_b)
            print(f"    ✓ {file:40} | {size_display:10}")
    
    total_all += model_total
    print(f"  ✓ Subtotal: {format_size(model_total)}")

print(f"\n" + "="*80)
print(f"📊 GRAND TOTAL: {format_size(total_all)}")
print("="*80)

print(f"\n🔍 CHECKING FOR EMPTY FOLDERS:\n")

empty_folders = []
for folder_name, folder_path in FOLDERS.items():
    if os.path.exists(folder_path):
        files = [f for f in os.listdir(folder_path) if os.path.isfile(os.path.join(folder_path, f))]
        if not files:
            empty_folders.append((folder_name, folder_path))
            print(f"  ⚠️  EMPTY: {folder_name:20} → {folder_path}")

if not empty_folders:
    print(f"  ✓ No empty folders - all systems ready!\n")
else:
    print(f"\n  Note: {len(empty_folders)} empty folders can be deleted if not needed\n")

print("="*80)
print("✅ ALL DOWNLOADS COMPLETE!")
print("="*80)
print(f"\n✓ Total Models Downloaded: {len([f for f in model_folders.values() if os.path.exists(f) and os.listdir(f)])}")
print(f"✓ Total Storage Used: {format_size(total_all)}")
print(f"✓ All data saved to Google Drive: {MASTER_DIR}")
print(f"\n🎯 NEXT STEPS:")
print(f"   1. Run Notebook 05: Text-to-Image Generation")
print(f"   2. Run Notebook 02: Advanced Features")
print(f"   3. Run Notebook 03: Character Analysis")
print(f"   4. Run Notebook 04: ComfyUI Dashboard (Optional)")
print("\n" + "="*80 + "\n")